# Ajuste de hiperparámetros y variables por resultado

## Qué responde este notebook

[`4.2_modelos_recepciones_yardas.ipynb`](4.2_modelos_recepciones_yardas.ipynb) y
[`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb) eligieron XGBoost para los tres
resultados con una grilla pequeña (profundidad, número de árboles y tasa de aprendizaje). Aquí no se
vuelven a comparar familias; se prueba si el mismo XGBoost mejora por dos caminos, cada uno por
separado:

1. **Hiperparámetros**: una búsqueda compacta en la que cada combinación encuentra su número de
   árboles con *early stopping*, usando la misma métrica que decide (RMSE o deviance de Poisson). En
   touchdowns se busca además la potencia de Tweedie, comparada contra el objetivo Poisson.
2. **Variables propias por resultado**: en lugar de las 34 compartidas (sección 8 del
   [glosario](../../../docs/data/GLOSARIO_VARIABLES.md)), solo las 15 más importantes
   de cada resultado según [`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb), en la
   corrida con la que se fijaron las 34 (ver la sección 4.2 de ese notebook).

Un cambio se adopta solo si mejora la métrica principal en validación **y** la mejora es real: el
intervalo bootstrap por semanas de la diferencia (ver
[`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb)) no incluye cero. La
búsqueda es compacta para acotar el cómputo; `experimentos.ESPACIO_HIPERPARAMETROS_XGBOOST` define un
espacio más amplio (regularización y submuestreo) por si se quiere explorar más.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

from itertools import product

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

wr = features.construir_tabla_modelado(range(2016, 2026))
features_compartidas = features.columnas_por_modelo("arbol")
train_mask, val_mask, test_mask = experimentos.dividir_temporal(
    wr, range(2016, 2022), [2022, 2023], [2024, 2025]
)
TARGETS = list(modeling.TARGETS)

def evaluar(target, columnas, params):
    modelo = XGBRegressor(**params, random_state=0, n_jobs=-1)
    return experimentos.evaluar_modelo(modelo, wr[columnas], wr[target], train_mask, val_mask, test_mask)

def comparar(target, pred_nueva, pred_actual):
    return experimentos.diferencia_bootstrap(
        wr[val_mask], wr.loc[val_mask, target], pred_nueva[val_mask], pred_actual[val_mask],
        modeling.METRICA_PRINCIPAL[target])

In [2]:
FEATURES_PROPIAS = {
    "receptions": [
        "target_share_last5_avg", "receptions_season_avg", "depth_team", "receptions_last3_avg",
        "receptions_last5_avg", "receptions_career_avg", "targets_season_avg", "targets_last5_avg",
        "draft_pick", "receiving_10_last3_avg", "receiving_epa_last3_avg", "targets_last3_avg",
        "target_share_career_avg", "receiving_20_last3_avg", "target_share_last3_avg",
    ],
    "receiving_yards": [
        "target_share_last5_avg", "depth_team", "wopr_last5_avg", "fantasy_points_ppr_last5_avg",
        "targets_season_avg", "receptions_season_avg", "receiving_yards_season_avg",
        "receiving_first_downs_last5_avg", "wopr_last3_avg", "targets_last5_avg",
        "air_yards_share_season_avg", "receiving_yards_career_avg", "receiving_yards_last3_avg",
        "draft_pick", "receiving_air_yards_season_avg",
    ],
    "receiving_tds": [
        "targets_last5_avg", "air_yards_share_career_avg", "draft_pick", "fantasy_points_ppr_last5_avg",
        "receiving_yards_career_avg", "receiving_first_downs_career_avg", "receiving_10_last3_avg",
        "depth_team", "air_yards_share_last3_avg", "target_share_last3_avg",
        "fantasy_points_ppr_season_avg", "receiving_tds_season_avg", "receiving_16_last5_avg",
        "catch_rate_career_avg", "fantasy_points_ppr_last3_avg",
    ],
}
for target, cols in FEATURES_PROPIAS.items():
    faltantes = [c for c in cols if c not in wr.columns]
    print(f"{target}: {len(cols)} columnas propias, faltantes: {faltantes}")

receptions: 15 columnas propias, faltantes: []
receiving_yards: 15 columnas propias, faltantes: []
receiving_tds: 15 columnas propias, faltantes: []


## 1. El punto de partida

El XGBoost elegido en 4.2 y 4.3: la misma grilla de 12 combinaciones, elegida por la métrica
principal en validación. En touchdowns, con objetivo `count:poisson`, el que quedó primero.

In [3]:
GRILLA = list(product((3, 4, 6), (200, 400), (0.03, 0.08)))
OBJETIVO = {"receptions": {}, "receiving_yards": {}, "receiving_tds": {"objective": "count:poisson"}}

punto_partida = {}
for target in TARGETS:
    criterio = modeling.METRICA_PRINCIPAL[target]
    mejor = None
    for d, n, lr in GRILLA:
        params = {**OBJETIVO[target], "max_depth": d, "n_estimators": n, "learning_rate": lr}
        salida = evaluar(target, features_compartidas, params)
        if mejor is None or salida["val"][criterio] < mejor["salida"]["val"][criterio]:
            mejor = {"params": params, "salida": salida}
    punto_partida[target] = mejor
    print(f"{target}: {mejor['params']} | {criterio} de validacion = {mejor['salida']['val'][criterio]:.4f}")

receptions: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03} | rmse de validacion = 1.9293


receiving_yards: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03} | rmse de validacion = 29.5718


receiving_tds: {'objective': 'count:poisson', 'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.08} | deviance_poisson de validacion = 0.6167


## 2. Ajuste compacto de hiperparámetros

- **Recepciones y yardas**: profundidad 3, 4 o 5 y tasa 0.03 o 0.05 (6 combinaciones), con early
  stopping sobre RMSE de validación.
- **Touchdowns**: potencia de Tweedie 1.2 a 1.8 y profundidad 2 a 4 (15 combinaciones), más el
  objetivo Poisson con profundidad 2 a 4 (3 combinaciones), con tasa 0.03 y early stopping sobre la
  log-verosimilitud de Poisson, que equivale a la deviance salvo una constante.

In [4]:
ajustados = {}
for target in TARGETS:
    criterio = modeling.METRICA_PRINCIPAL[target]
    busquedas = [experimentos.buscar_hiperparametros(wr, target, features_compartidas, train_mask, val_mask)]
    if target == "receiving_tds":
        busquedas.append(experimentos.buscar_hiperparametros(
            wr, target, features_compartidas, train_mask, val_mask,
            espacio={"max_depth": [2, 3, 4]}, params_fijos={"objective": "count:poisson"}))
    modelo, config, _ = min(busquedas, key=lambda b: b[2][f"val_{criterio}"].iloc[0])
    pred = pd.Series(modeling.clip_no_negativo(modelo.predict(wr[features_compartidas])), index=wr.index)
    ajustados[target] = {"config": config, "pred": pred,
                         "intentos": pd.concat([b[2] for b in busquedas], ignore_index=True)}
    print(f"{target}: mejor combinacion {{" + ", ".join(f"{k}={config[k]}" for k in
          ("objective", "tweedie_variance_power", "max_depth", "learning_rate", "n_estimators") if k in config) + "}")

columnas_intentos = ["objective", "tweedie_variance_power", "max_depth", "learning_rate", "n_estimators_optimo",
                     "val_deviance_poisson", "val_d2_oos", "val_sesgo", "val_pendiente_calibracion"]
ajustados["receiving_tds"]["intentos"].reindex(columns=columnas_intentos).sort_values("val_deviance_poisson").head(8).round(4)

receptions: mejor combinacion {max_depth=4, learning_rate=0.05, n_estimators=80}


receiving_yards: mejor combinacion {max_depth=3, learning_rate=0.03, n_estimators=224}


receiving_tds: mejor combinacion {objective=reg:tweedie, tweedie_variance_power=1.8, max_depth=2, learning_rate=0.03, n_estimators=196}


,objective,tweedie_variance_power,max_depth,learning_rate,n_estimators_optimo,val_deviance_poisson,val_d2_oos,val_sesgo,val_pendiente_calibracion
0,reg:tweedie,1.80,2,0.03,196,0.6161,0.1231,0.0183,1.0366
1,reg:tweedie,1.65,2,0.03,172,0.6165,0.1226,0.0203,1.0368
2,reg:tweedie,1.35,3,0.03,189,0.6166,0.1224,0.0206,0.9842
3,reg:tweedie,1.35,4,0.03,162,0.6166,0.1224,0.0171,0.9690
4,reg:tweedie,1.20,3,0.03,234,0.6166,0.1223,0.0228,0.9632
5,reg:tweedie,1.50,3,0.03,163,0.6168,0.1221,0.0186,0.9935
15,count:poisson,NaN,3,0.03,546,0.6168,0.1220,0.0265,0.9508
6,reg:tweedie,1.50,2,0.03,209,0.6170,0.1218,0.0218,1.0059


In [5]:
filas = {}
for target in TARGETS:
    criterio = modeling.METRICA_PRINCIPAL[target]
    actual = punto_partida[target]["salida"]
    medida_nueva = modeling.metricas_target(target, wr.loc[val_mask, target], ajustados[target]["pred"][val_mask],
                                            media_entrenamiento=actual["media_entrenamiento"])
    c = comparar(target, ajustados[target]["pred"], actual["pred"])
    ajustados[target]["adoptar"] = (not c["empate"]) and c["diferencia"] < 0
    filas[target] = {"criterio": criterio, "punto de partida": actual["val"][criterio], "ajustado": medida_nueva[criterio],
                     "diferencia": c["diferencia"], "IC 95% inferior": c["ic_inferior"], "IC 95% superior": c["ic_superior"],
                     "se adopta": ajustados[target]["adoptar"]}
pd.DataFrame(filas).T

,criterio,punto de partida,ajustado,diferencia,IC 95% inferior,IC 95% superior,se adopta
receptions,rmse,1.929261,1.928129,-0.001132,-0.007123,0.004514,False
receiving_yards,rmse,29.571793,29.56971,-0.002083,-0.016974,0.013672,False
receiving_tds,deviance_poisson,0.616653,0.616099,-0.000554,-0.004451,0.003184,False


**Lectura.** El ajuste compacto mejora la métrica principal en milésimas en los tres resultados
(recepciones −0.0011 de RMSE, yardas −0.0021, touchdowns −0.0006 de deviance), y en los tres el
intervalo incluye cero: ninguna mejora es real, así que no se adopta. Cada resultado compara la mejor
de varias combinaciones, así que el ajuste por comparaciones múltiples de 4.1 solo ensancharía esos
intervalos y la conclusión no cambia.

En touchdowns la mejor combinación fue Tweedie con potencia 1.8 y profundidad 2, pero las ocho
mejores quedan entre 0.6161 y 0.6170, prácticamente igual que el punto de partida con objetivo
Poisson (0.6167): con estas variables, la grilla original ya queda en lo mejor que encuentra la
búsqueda.

## 3. ¿Un conjunto de variables propio por resultado?

Mismos hiperparámetros del punto de partida, cambiando solo qué variables ve el modelo: las 15 más
importantes de cada resultado en lugar de las 34 compartidas.

In [6]:
filas = {}
propias = {}
for target in TARGETS:
    criterio = modeling.METRICA_PRINCIPAL[target]
    actual = punto_partida[target]["salida"]
    propias[target] = evaluar(target, FEATURES_PROPIAS[target], punto_partida[target]["params"])
    c = comparar(target, propias[target]["pred"], actual["pred"])
    filas[target] = {"criterio": criterio, "34 compartidas": actual["val"][criterio],
                     "15 propias": propias[target]["val"][criterio], "diferencia": c["diferencia"],
                     "IC 95% inferior": c["ic_inferior"], "IC 95% superior": c["ic_superior"],
                     "se adopta": (not c["empate"]) and c["diferencia"] < 0}
pd.DataFrame(filas).T

,criterio,34 compartidas,15 propias,diferencia,IC 95% inferior,IC 95% superior,se adopta
receptions,rmse,1.929261,1.935073,0.005813,-0.000526,0.012551,False
receiving_yards,rmse,29.571793,29.597565,0.025773,-0.047541,0.099312,False
receiving_tds,deviance_poisson,0.616653,0.619566,0.002913,0.000417,0.005508,False


**Lectura.** Usar solo las 15 variables propias empeora la métrica en los tres resultados, y en
touchdowns el empeoramiento es real (+0.0029 de deviance, con intervalo de +0.0004 a +0.0055).
Se quedan las 34 compartidas: juntar lo importante para los tres resultados no le cuesta precisión a ninguno.

## 4. Configuración elegida por resultado

La que queda después de aplicar las dos reglas, con su revisión de requisitos en validación. La
prueba se reporta aquí por primera vez para estas configuraciones; la evaluación año por año está en
[`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb).

In [7]:
elegida = {}
for target in TARGETS:
    if ajustados[target]["adoptar"]:
        cfg = {k: v for k, v in ajustados[target]["config"].items() if k not in ("random_state", "n_jobs")}
        origen = "ajuste compacto"
    else:
        cfg = punto_partida[target]["params"]
        origen = "punto de partida"
    salida = evaluar(target, features_compartidas, cfg)
    elegida[target] = {"params": cfg, "origen": origen, "salida": salida}
    cumple, detalle = experimentos.cumple_requisitos(target, salida["val"])
    print(f"{target}: {origen} -> {cfg}")
    print(f"   requisitos en validacion: {'cumple' if cumple else 'NO cumple'}")
    for chequeo, ok in detalle.items():
        print(f"      {'ok ' if ok else 'NO '} {chequeo}")

columnas = ["rmse", "r2_oos", "deviance_poisson", "d2_oos", "sesgo", "pendiente_calibracion", "mae"]
pd.concat({parte: pd.DataFrame({t: {k: elegida[t]["salida"][parte].get(k) for k in columnas} for t in TARGETS}).T
           for parte in ("val", "test")}, axis=1).astype(float).round(4)

receptions: punto de partida -> {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03}
   requisitos en validacion: cumple
      ok  R2 fuera de muestra > 0 (validacion)
      ok  sesgo adicional al de la media de entrenamiento <= 5% de la media (validacion)
      ok  pendiente de calibracion entre 0.9 y 1.1 (validacion)


receiving_yards: punto de partida -> {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03}
   requisitos en validacion: cumple
      ok  R2 fuera de muestra > 0 (validacion)
      ok  sesgo adicional al de la media de entrenamiento <= 5% de la media (validacion)
      ok  pendiente de calibracion entre 0.9 y 1.1 (validacion)


receiving_tds: punto de partida -> {'objective': 'count:poisson', 'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.08}
   requisitos en validacion: cumple
      ok  R2 fuera de muestra > 0 (validacion)
      ok  D2 fuera de muestra > 0 (validacion)
      ok  sesgo adicional al de la media de entrenamiento <= 5% de la media (validacion)
      ok  pendiente de calibracion entre 0.9 y 1.1 (validacion)


val                                                                             test                                           \
                    rmse  r2_oos deviance_poisson  d2_oos   sesgo pendiente_calibracion      mae     rmse  r2_oos deviance_poisson  d2_oos   sesgo   
receptions        1.9293  0.4377           1.4444  0.4316  0.0971                1.0494   1.4491   1.8258  0.4585           1.3913  0.4520  0.0639   
receiving_yards  29.5718  0.3624              NaN     NaN  1.5727                1.0711  22.0045  27.9143  0.3754              NaN     NaN  1.1826   
receiving_tds     0.4347  0.0826           0.6167  0.1223  0.0261                0.9604   0.3081   0.4463  0.0877           0.6252  0.1355  0.0001   

                                                
                pendiente_calibracion      mae  
receptions                     1.0245   1.3641  
receiving_yards                1.0300  20.5522  
receiving_tds                  1.0104   0.3045

**Lectura.** Quedan las configuraciones de partida: XGBoost de profundidad 3, 200 árboles y tasa
0.03 para recepciones y yardas, y XGBoost con objetivo Poisson, profundidad 3, 200 árboles y tasa
0.08 para touchdowns. Las tres cumplen los requisitos en validación. En prueba quedan al nivel de
validación o un poco arriba: R² fuera de muestra de 0.4585 en recepciones (0.4377 en validación),
0.3754 en yardas (0.3624) y 0.0877 en touchdowns (0.0826), con D² de 0.1355 en touchdowns
(0.1223) y pendientes de calibración entre 1.01 y 1.03. En touchdowns el sesgo de prueba es prácticamente nulo (+0.0001).

## Cierre

Ni un ajuste más fino de hiperparámetros ni variables propias por resultado mejoran de forma real a
XGBoost con la grilla original. Quedan esas configuraciones, que cumplen los requisitos. Los
notebooks 4.5 a 4.9 prueban alternativas de modelado contra ellas, y la etapa 5 revisa su estabilidad
año por año, las compara contra el flujo que existía antes de este proyecto y explica sus predicciones.

Anterior: [`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb) · Siguiente:
[`4.5_regresion_cuantiles.ipynb`](4.5_regresion_cuantiles.ipynb)